In [3]:
import pandas as pd
from datetime import datetime

# 1. Read invoice data from CSV
df = pd.read_csv('raw_invoices.csv')

# Ensure dates are parsed correctly
df['Invoice_Date'] = pd.to_datetime(df['Invoice_Date'])
df['Due_Date'] = pd.to_datetime(df['Due_Date'])

# 2. Calculate Total Amount per line item
df['Line_Total'] = df['Unit_Price'] * df['Quantity']

# 3. Generate a consolidated invoice report
# Grouping multiple items belonging to the same invoice into a single row
consolidated = df.groupby(['Invoice_Number', 'Customer_Name', 'Invoice_Date', 'Due_Date']).agg(
    Total_Items=('Quantity', 'sum'),
    Invoice_Total=('Line_Total', 'sum')
).reset_index()



In [6]:
# 4. Identify Overdue Invoices
# Compares the due date against the current system date
current_date = pd.to_datetime('today')
consolidated['Status'] = consolidated['Due_Date'].apply(
    lambda x: 'Overdue ' if x < current_date else 'Pending '
)

# Sort by Due Date to prioritize the most urgent invoices
consolidated = consolidated.sort_values(by='Due_Date').reset_index(drop=True)

# 5. Export the final report as CSV
output_filename = 'consolidated_invoice_report.csv'
consolidated.to_csv(output_filename, index=False)

print("---  CONSOLIDATED INVOICE REPORT ---")
print(consolidated[['Invoice_Number', 'Customer_Name', 'Invoice_Total', 'Status']].to_string(index=False))
print(f"\n Final report exported successfully to '{output_filename}'")

---  CONSOLIDATED INVOICE REPORT ---
Invoice_Number     Customer_Name  Invoice_Total   Status
      INV-1005        Wayne Tech         506.80 Overdue 
      INV-1012 Global Industries        4388.91 Overdue 
      INV-1011 Global Industries         514.00 Overdue 
      INV-1004        Wayne Tech        2597.16 Overdue 
      INV-1018          TechNova        5128.26 Overdue 
      INV-1019         Acme Corp        4991.69 Overdue 
      INV-1001          TechNova        2642.10 Overdue 
      INV-1014          TechNova        3006.64 Overdue 
      INV-1015        Wayne Tech         949.90 Overdue 
      INV-1010 Global Industries        2430.20 Overdue 
      INV-1016 Global Industries        8502.11 Pending 
      INV-1009        Wayne Tech        5049.46 Pending 
      INV-1002 Stark Enterprises        3874.52 Pending 
      INV-1017         Acme Corp         353.78 Pending 
      INV-1008 Global Industries        4299.50 Pending 
      INV-1007 Stark Enterprises        5316.23 Pen